## Includes

In [46]:
import os
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
import torch.utils.data as data
import lightning as L

from vv.utilities.utilities import get_dir

## Model setup

In [47]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Sequential(nn.Linear(28 * 28, 64), nn.ReLU(), nn.Linear(64, 3))

    def forward(self, x):
        return self.l1(x)


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 28 * 28))

    def forward(self, x):
        return self.l1(x)

In [48]:
class LitAutoEncoder(L.LightningModule):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

        # Saves hyperparameters that are passed to the __init__ method
        self.save_hyperparameters(ignore=['encoder', 'decoder'])

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        x, y = batch
        x = x.view(x.size(0), -1)
        z = self.encoder(x)
        x_hat = self.decoder(z)
        loss = F.mse_loss(x_hat, x)
        return loss
    
    def validation_step(self, batch, batch_idx):
        # this is the validation loop
        x, y = batch
        x = x.view(x.size(0), -1)
        z = self.encoder(x)
        x_hat = self.decoder(z)
        val_loss = F.mse_loss(x_hat, x)
        self.log("val_loss", val_loss)
    
    def test_step(self, batch, batch_idx):
        # this is the test loop
        x, y = batch
        x = x.view(x.size(0), -1)
        z = self.encoder(x)
        x_hat = self.decoder(z)
        test_loss = F.mse_loss(x_hat, x)
        self.log("test_loss", test_loss)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        return optimizer

## Data setup

In [49]:
# Load data sets
transform = transforms.ToTensor()
train_set = MNIST(root=get_dir("data"), download=True, train=True, transform=transform);
test_set = MNIST(root=get_dir("data"), download=True, train=False, transform=transform);

# use 20% of training data for validation
train_set_size = int(len(train_set) * 0.8)
valid_set_size = len(train_set) - train_set_size

# split the train set into two
seed = torch.Generator().manual_seed(42)
train_set, valid_set = data.random_split(train_set, [train_set_size, valid_set_size], generator=seed)

In [50]:
# model
model = LitAutoEncoder(Encoder(), Decoder())

train_loader = DataLoader(train_set, batch_size=5000, num_workers=10)
valid_loader = DataLoader(valid_set, batch_size=5000, num_workers=10)
test_loader = DataLoader(test_set, batch_size=5000, num_workers=10)

## Training process

In [51]:
# train model with validation
trainer = L.Trainer(default_root_dir=get_dir("models", create=True),
                    max_epochs=100,
                    enable_checkpointing=True)
trainer.fit(model=model,
            train_dataloaders=train_loader,
            val_dataloaders=valid_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



  | Name    | Type    | Params
------------------------------------
0 | encoder | Encoder | 50.4 K
1 | decoder | Decoder | 51.2 K
------------------------------------
101 K     Trainable params
0         Non-trainable params
101 K     Total params
0.407     Total estimated model params size (MB)


Epoch 6: 100%|██████████| 10/10 [00:00<00:00, 12.15it/s, v_num=4]           

### Continue training

In [53]:
model_loaded = LitAutoEncoder.load_from_checkpoint(
    checkpoint_path=get_dir("models",
                            "lightning_logs",
                            "version_4",
                            "checkpoints",
                            "epoch=5-step=60.ckpt"),
    # The following arguments are only necessary if hyperparams of model are not saved
    # larger stuff like nn modules should not be saved
    # In this case they need to be passed as arguments to the load_from_checkpoint method
    encoder=Encoder(),
    decoder=Decoder()
)

This only restores the weights of the model. In order to restore all training parameters like e.g. scheduler, do the following:

In [55]:
# model
model = LitAutoEncoder(Encoder(), Decoder())
trainer = L.Trainer(default_root_dir=get_dir("models", create=True),
                    max_epochs=100,
                    enable_checkpointing=True)

# automatically restores model, epoch, step, LR schedulers, etc...
trainer.fit(model,
            train_dataloaders=train_loader,
            val_dataloaders=valid_loader,
            ckpt_path=get_dir("models",
                              "lightning_logs",
                              "version_4",
                              "checkpoints",
                              "epoch=5-step=60.ckpt"))

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Restoring states from the checkpoint path at /home/computerman/Desktop/Master DCS/Semester 3/Praktikum/VQ-VAE-Praktikum-SS24-HD/models/lightning_logs/version_4/checkpoints/epoch=5-step=60.ckpt
/home/computerman/Desktop/Master DCS/Semester 3/Praktikum/VQ-VAE-Praktikum-SS24-HD/venv/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:361: The dirpath has changed from '/home/computerman/Desktop/Master DCS/Semester 3/Praktikum/VQ-VAE-Praktikum-SS24-HD/models/lightning_logs/version_4/checkpoints' to '/home/computerman/Desktop/Master DCS/Semester 3/Praktikum/VQ-VAE-Praktikum-SS24-HD/models/lightning_logs/version_6/checkpoints', therefore `best_model_score`, `kth_best_model_path`, `kth_value`, `last_model_path` and `best_k_models` won't be reloaded. Only `best_model_path` will be reloaded.
LOCAL_RANK: 0 - CUDA_VISIBLE_D

/home/computerman/Desktop/Master DCS/Semester 3/Praktikum/VQ-VAE-Praktikum-SS24-HD/venv/lib/python3.11/site-packages/lightning/pytorch/loops/fit_loop.py:298: The number of training batches (10) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 99: 100%|██████████| 10/10 [00:01<00:00,  6.67it/s, v_num=6]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 10/10 [00:01<00:00,  6.63it/s, v_num=6]


## Testing

In [58]:
model.eval()
trainer.test(model, dataloaders=test_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing DataLoader 0: 100%|██████████| 2/2 [00:00<00:00, 172.22it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    0.04031293839216232    │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.04031293839216232}]